# Promptable transcription

The Whisper and WhisperX notebook stores a video, extracts the audio, and transcribes it with a computed column. Each model was its own column, and a long file had to be split first.

[Nemotron 3.5 Transcribe](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B) is one checkpoint on Hugging Face. It takes the recording and a prompt. The prompt is a column, so verbatim, cleaned, and name-aware transcripts are three rows, not three models.

Start Jupyter with `uv run jupyter lab` in this folder. The Hugging Face token needs Inference Providers credits. See the README.

In [ ]:
!uv sync

In [ ]:
import pixeltable as pxt

# Start fresh: drop the tables from any earlier run.
pxt.drop_dir('nemotron_demo', force=True, if_not_exists='ignore')

## The table

`video` is what you insert. `audio` is extracted from it. `transcript` sends that audio, plus `prompt`, to Nemotron.

```mermaid
flowchart LR
  video --> audio["audio"]
  audio --> transcript
  prompt --> transcript["transcript"]
```

In [ ]:
!cat schema.py

`pxt schema diff` prints the plan. `pxt schema update` applies it.

In [ ]:
!pxt schema diff schema.py nemotron_demo

In [ ]:
!pxt schema update schema.py nemotron_demo -f

In [ ]:
!pxt ls nemotron_demo

## Three prompts, one clip

The clip is the short Lex Fridman excerpt from the transcription cookbook. Each row is one request to Hugging Face. The prompts are the ones on the model card: verbatim, cleaned, and a name to keep spelled as given.

In [ ]:
CLIP = (
    'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/'
    'audio-transcription-demo/Lex-Fridman-Podcast-430-Excerpt-0.mp4'
)

recordings = pxt.get_table('nemotron_demo.recordings')
recordings.insert([
    {
        'label': 'verbatim',
        'video': CLIP,
        'prompt': (
            'Audio transcription. Language: as spoken. '
            'Wording: verbatim, including fillers and repetitions. '
            'Formatting: punctuation and capitalization. Output: transcript only.'
        ),
    },
    {
        'label': 'clean',
        'video': CLIP,
        'prompt': (
            'Audio transcription. Language: as spoken. '
            'Wording: omit fillers and accidental repetitions; '
            'keep the remaining spoken wording unchanged. '
            'Formatting: punctuation and capitalization. '
            'Numbers in their standard written form. Output: transcript only.'
        ),
    },
    {
        'label': 'names',
        'video': CLIP,
        'prompt': (
            'Transcribe with punctuation and capitalization. '
            'Context terms: Lex Fridman.'
        ),
    },
])

In [ ]:
recordings.select(recordings.label, recordings.prompt, recordings.transcript).collect()